# Sanskrit Studio - batch render
**Not yet run on real Colab by the author.** See COLAB.md, section 'Batch flow'.

Runtime > Change runtime type > T4 GPU, then run the single cell below and pick `batch.zip` from the app when asked. Results are written to your Drive (`batch_results/...`); import them in the app from the Drive folder, or set `DOWNLOAD_ZIP` to also download `results.zip`. If the session drops, run the cell again with the same `batch.zip` to resume.

In [ ]:
#@title Sanskrit Studio - batch render (run this one cell) { display-mode: "form" }
DRIVE_FOLDER = "sanskrit_studio"  #@param {type:"string"}
WORKERS = 1  #@param {type:"integer"}
DOWNLOAD_ZIP = False  #@param {type:"boolean"}

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path, PurePosixPath

# ---- 1. GPU check ---------------------------------------------------------
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise SystemExit("NO GPU. Runtime > Change runtime type > T4 GPU, then run this cell again.")

# ---- 2. Mount Drive -------------------------------------------------------
from google.colab import drive, files
drive.mount("/content/drive")
R = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
print("R =", R)

# ---- 3. Verify required files ---------------------------------------------
need = ["models/voice_steer_ema_2026-06-17.pt", "models/voice_armA_ema_2026-06-11.pt",
        "models/voc_bigvgan_EMA_2026-06-11.pth", "models/vocab.txt",
        "code/backend/__init__.py", "code/vagdhenu/src", "code/BigVGAN/configs"]
missing = [p for p in need if not (Path(R) / p).exists()]
for p in need:
    print("OK     " if p not in missing else "MISSING", p)
if missing:
    raise SystemExit("Missing files above. See COLAB.md one-time setup (steps 1-2).")

# ---- 4. Install dependencies (torch is preinstalled; do not reinstall it) --
try:
    import f5_tts  # noqa: F401
    print("f5_tts already installed; skipping pip install")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "f5-tts @ git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7",
        "vocos", "x-transformers>=2.19.7", "indic-transliteration", "librosa", "soundfile>=0.12.1",
        "safetensors", "huggingface_hub", "pyarrow<20", "scipy", "python-dotenv"], check=True)
import torch, torchaudio
print("torch", torch.__version__, "torchaudio", torchaudio.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "torch sees no CUDA"

# ---- 5. Copy code locally, symlink models ---------------------------------
APP = "/content/app"
shutil.rmtree(APP, ignore_errors=True)
shutil.copytree(f"{R}/code", APP)
os.makedirs(f"{APP}/vagdhenu", exist_ok=True)
if not os.path.exists(f"{APP}/vagdhenu/models"):
    os.symlink(f"{R}/models", f"{APP}/vagdhenu/models")   # weights stay on Drive
os.environ["SS_VAGDHENU"] = f"{APP}/vagdhenu"
os.environ["SS_BIGVGAN"] = f"{APP}/BigVGAN"
os.environ["SS_STORAGE"] = "/content/ss_storage"
os.chdir(APP)
if APP not in sys.path:
    sys.path.insert(0, APP)
MODELS = f"{R}/models"

# ---- 6. Smoke import ------------------------------------------------------
from backend.engine import synth
print("engine import OK;", torch.cuda.get_device_name(0))

# ---- 7. Upload batch.zip and extract it safely ----------------------------
print("Choose batch.zip (downloaded from the app) ...")
up = files.upload()
if not up:
    raise SystemExit("No file uploaded. Run the cell again and pick batch.zip.")
name = next(iter(up))
B = "/content/batch"
shutil.rmtree(B, ignore_errors=True)   # results (below) are kept, so a rerun resumes
with zipfile.ZipFile(name) as z:
    total = 0
    for i in z.infolist():
        p = PurePosixPath(i.filename.replace("\\", "/"))
        if p.is_absolute() or ".." in p.parts or (len(i.filename) > 1 and i.filename[1] == ":"):
            raise SystemExit(f"Unsafe path in zip: {i.filename!r}")
        total += i.file_size
        if total > 500 * 1024 * 1024:
            raise SystemExit("batch.zip is too large when unpacked")
    z.extractall(B)
if not os.path.exists(f"{B}/batch.json"):
    raise SystemExit("batch.json not found in the zip. Is this the batch.zip from the app?")
meta = json.load(open(f"{B}/batch.json"))
n_jobs = len(meta["jobs"])
batch_id = str(meta.get("batch_id") or "batch")

def safe_rel(p, what):
    t = str(p or "")
    if (not t or "\\" in t or t.startswith("/") or ":" in t
            or any(x in ("", "..") for x in t.split("/"))):
        raise SystemExit(f"Unsafe {what} in batch.json: {t!r}. Regenerate the batch in the app.")
    return t
BATCH_DIR = safe_rel(meta.get("batch_dir"), "batch_dir")
RESULTS_ROOT = safe_rel(meta.get("results_root"), "results_root")
job_dirs = {str(j["job_id"]): safe_rel(j.get("result_dir"), "result_dir") for j in meta["jobs"]}
print(f"batch {batch_id} has {n_jobs} job(s)")

# Code shipped inside the zip wins over the Drive copy (Drive code/ is only a fallback).
if os.path.isdir(f"{B}/code/backend"):
    shutil.rmtree(f"{APP}/backend", ignore_errors=True)
    shutil.copytree(f"{B}/code/backend", f"{APP}/backend")
    print("Using code from batch.zip; code_version =", meta.get("code_version"))
else:
    print("NOTE: batch.zip has no code/backend; using the Drive copy of the code (may be stale).")

# ---- 8. Run the worker (results live on Drive so a wiped VM can resume) ---
# Layout comes from batch.json: <R>/<result_dir>/chant.wav + result.json per verse.
OUT = f"{R}/{BATCH_DIR}"
os.makedirs(OUT, exist_ok=True)
WORKERS = max(1, int(WORKERS))
def is_done(jid):
    d = Path(R) / job_dirs[jid]
    return (d / "chant.wav").exists() and (d / "result.json").exists()
done_before = sum(1 for j in job_dirs if is_done(j))
t_start = time.time()
proc = subprocess.Popen([sys.executable, "-u", "-m", "backend.worker.colab_worker",
        "--batch", B, "--results-root", R, "--models-dir", MODELS, "--device", "cuda",
        "--workers", str(WORKERS)],
        cwd=APP, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
except BaseException:
    proc.terminate()   # the worker's parent then terminates its children
    try: proc.wait(30)
    except subprocess.TimeoutExpired: proc.kill()
    raise
elapsed = time.time() - t_start

# ---- 9. Build results.zip on Drive (layout results/<job_id>/...) -----------
ep = Path(OUT) / "errors.json"
errs = json.loads(ep.read_text()) if ep.exists() else {}
ok_ids = [j for j in job_dirs if is_done(j)]
n_ok = len(ok_ids)
n_new = n_ok - done_before
if n_ok == 0:
    print(f"No verses were rendered (worker exit code {rc}). Errors:")
    for k, v in errs.items():
        print(f"  job {k} failed: {v[:300]}")
    raise SystemExit("Nothing to zip. Fix the errors above and run the cell again.")
ZIP = f"{OUT}/results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for jid in ok_ids:
        for fn in ("chant.wav", "result.json"):
            z.write(Path(R) / job_dirs[jid] / fn, f"results/{jid}/{fn}")
    z.writestr("results/errors.json", json.dumps(errs))

# ---- 10. Summary ----------------------------------------------------------
print("=" * 50)
print(f"Finished: {n_ok} ok, {len(errs)} failed, {n_jobs} in batch (worker exit code {rc}, {WORKERS} worker(s))")
if n_new > 0:
    print(f"{n_new} rendered, {elapsed / n_new:.1f} s/verse incl. model load (run took {elapsed:.0f}s)")
else:
    print("nothing new to render")
print(f"Results are on Drive under My Drive/{DRIVE_FOLDER}/ :")
for d in sorted({str(PurePosixPath(job_dirs[j]).parent) for j in ok_ids}):
    print(f"  {d}/")
print(f"  {BATCH_DIR}/results.zip  (and errors.json)")
for k, v in errs.items():
    print(f"  job {k} failed: {v[:200]}")
print("Next: in the app: Generate > Epic verses > your batch > Import from Drive folder (needs the Drive "
      "folder mounted locally), or download results.zip from Drive and upload it.")
if rc != 0:
    print("Worker exited abnormally; if it was a disconnect, run this cell again with the same batch.zip to resume.")
if DOWNLOAD_ZIP:
    files.download(ZIP)
